In [ ]:
%python -m pip install libsqlite3-dev  -U --force-reinstall

In [3]:
#yum install sqlite-devel
%pip install matplotlib
%pip install panda

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import matplotlib.pyplot as plt 
import os

In [5]:
p = os.getcwd()
PROJECT_PATH = os.path.join(p.split('change_aware_utg')[0],'change_aware_utg')
print(PROJECT_PATH)
BENCHMARK_PATH = os.path.join(PROJECT_PATH, "UTG_python_external_benchmark_light")
print(BENCHMARK_PATH)

/mnt/efs/people/rabaisha/GitLab/change_aware_utg
/mnt/efs/people/rabaisha/GitLab/change_aware_utg/UTG_python_external_benchmark_light


In [6]:
#Core Files consolidating all the data
#PROJECT_PATH = "/Users/rabaisha/GitLab/change_aware_utg"


fm_stat="bedrock_experiment/focal_method_statistics/Results/properties_of_fm.csv"
#Similar to the previous result, I also keep all cyclomatic complexity and line numbers in the project specific csv.
#https://gitlab.aws.dev/rabaisha/change_aware_utg/-/tree/master/bedrock_experiment/focal_method_statistics/Results?ref_type=heads
test_stat="test_analysis/Results/1072_tests_with_Focal_Methods_with_code_coverage.csv"
#test_stat="~/Downloads/1072_tests_with_Focal_Methods_with_code_coverage.csv"

fm_test_stat="bedrock_experiment/Results/Combined_result_of_fm_and_tests.csv" 

In [7]:
def if_file_exists(file_path):
    if os.path.exists(file_path):
        print(f"The file {file_path} exists.")
        return True
    else:
        print(f"The file {file_path} does not exist.")
        return False

In [8]:
fm_stat_path = os.path.join(PROJECT_PATH,fm_stat) 
assert(if_file_exists(fm_stat_path)==True)

test_stat_path = os.path.join(PROJECT_PATH,test_stat) 
assert(if_file_exists(test_stat_path)==True)

fm_test_stat_path = os.path.join(PROJECT_PATH,fm_test_stat) 
assert(if_file_exists(fm_test_stat_path)==True)# Read a CSV file from a URL

df_fm   = pd.read_csv(fm_stat_path)
df_test = pd.read_csv(test_stat_path)
df_both = pd.read_csv(fm_test_stat_path)

The file /mnt/efs/people/rabaisha/GitLab/change_aware_utg/bedrock_experiment/focal_method_statistics/Results/properties_of_fm.csv exists.
The file /mnt/efs/people/rabaisha/GitLab/change_aware_utg/test_analysis/Results/1072_tests_with_Focal_Methods_with_code_coverage.csv exists.
The file /mnt/efs/people/rabaisha/GitLab/change_aware_utg/bedrock_experiment/Results/Combined_result_of_fm_and_tests.csv exists.


Following Code Analyze The light benchmark (df_both, where all the test cases are passing). 
1. First we analyse the project level characteristics
2. We will analyze how many test cases cover a single FM
3. Test case statistics

In [ ]:
'''
Project Level Charachteristics
'''
unique_samples=df_both[['proj_name','test_filename','test_method','fm_filename','fm_method']]#.drop_duplicates().values

unique_project=df_both[['proj_name']].drop_duplicates().values
print('Number of Projects', len(unique_project))

grouped = unique_samples.groupby(['proj_name'])

per_project_tests_data = []

# Iterate over the groups and append their data to the list
for name, group in grouped:
    group_count = len(group)
    group_dict = {'Project': name, 'Count': group_count}
    per_project_tests_data.append(group_dict)

# Create a DataFrame from the group data
per_project_tests_df = pd.DataFrame(per_project_tests_data)
print(per_project_tests_df.head(22))


In [ ]:
print(per_project_tests_df['Count'].describe())
quantiles = per_project_tests_df['Count'].quantile([0.75, 0.80, 0.85, 0.9, 0.95, 1], interpolation='nearest')
print(quantiles)

In [ ]:
per_project_tests_df.hist(bins=20)
plt.xlabel("#Test Cases Per Project")
plt.ylabel("Frequency")
plt.show() 

In [9]:
'''We will analyze how many test cases cover a single FM''' 
unique_samples=df_both[['proj_name','test_filename','test_method','fm_filename','fm_method']]
print('Number of Test Cases', len(unique_samples))

grouped = unique_samples.groupby(['proj_name', 'fm_filename', 'fm_method'])#.size()

group_data = []

# Iterate over the groups and append their data to the list
for name, group in grouped:
    group_count = len(group)
    #group_dict = {'FM': name, 'Group': group.to_csv(index=False), 'Count': group_count}
    group_dict = {'FM': name, 'Count': group_count}
    group_data.append(group_dict)

# Create a DataFrame from the group data
per_fm_tests_df = pd.DataFrame(group_data)

# Write the DataFrame to a CSV file
#group_df.to_csv('groups.csv', index=False)


Number of Test Cases 690


In [10]:
print(per_fm_tests_df['Count'].describe())
quantiles = per_fm_tests_df['Count'].quantile([0.75, 0.80, 0.85, 0.9, 0.95, 1], interpolation='nearest')
print(quantiles)

count    350.000000
mean       1.971429
std        2.344117
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       25.000000
Name: Count, dtype: float64
0.75     2
0.80     3
0.85     3
0.90     4
0.95     5
1.00    25
Name: Count, dtype: int64


In [11]:
per_fm_tests_df.columns

Index(['FM', 'Count'], dtype='object')

In [ ]:
per_fm_tests_df.hist(bins=25)
plt.xlabel("#Test Cases Per Focal Method")
plt.ylabel("Frequency")
plt.show() 

In [ ]:
#Retrieve the statistics of the test cases present in the lite dataset
print(df_test.columns)
print(df_both.columns)
# print(df_fm.columns)

df_test_clean=df_test[['#proj_name', 'git_link', 'python_file_path', 'test_method_name',
       'claude_result', 'Static_Analysis_Result',
       'fm_source_file_by_static_analysis', 'Findings', 'All_Api_List',
       'fm_source_file_by_dynamic_analysis', 'covered_methods',
       'coverage_percentage', 'test_pass/fail']]

print(df_test_clean.columns)
df_test_clean = df_test_clean.rename(columns={'#proj_name': 'proj_name', 'python_file_path': 'test_filename', 'test_method_name': 'test_method'})
print(df_test_clean.columns)

In [ ]:
merged_df = pd.merge(df_both, df_test_clean, on=['proj_name','test_filename', 'test_method'], how='outer')
csv_file_path = os.path.join(BENCHMARK_PATH, "data/execution.csv")
# Save the DataFrame as a CSV file
merged_df.to_csv(csv_file_path, index=False)

In [ ]:
print(df_fm.columns)
df_test_clean = df_test_clean.rename(columns={'proj_name': 'Proj_name', 'test_filename': 'test_File_name', 'test_method': 'test_name'})
print(df_test_clean.columns)
df_repo=df_test_clean[['Proj_name', 'git_link', 'test_File_name']].drop_duplicates()
df_repo['test_File_name'] = df_repo['test_File_name'].apply(lambda x: pd.Series(x.split('/mnt/efs/people/urshanto/change_aware_utg/test_analysis/projects/')[1]))
df_repo['proj_path'] = df_repo['Proj_name'].apply(lambda x: pd.Series(os.path.join('/mnt/efs/people/urshanto/change_aware_utg/test_analysis/projects/',x)))

In [ ]:

print(df_repo.columns)
df_repo=df_repo[['Proj_name', 'git_link', 'proj_path']].drop_duplicates()
print(len(df_repo))

In [ ]:
csv_file_path = os.path.join(BENCHMARK_PATH, "data/repo.csv")
# Save the DataFrame as a CSV file
df_repo.to_csv(csv_file_path, index=False)

In [ ]:
import csv 
import subprocess

proj_dict = {}
# Open file  
with open(csv_file_path) as file_obj: 
    reader_obj = csv.reader(file_obj) 
    next(reader_obj)
    for row in reader_obj: 
        proj_path = row[2]
        command = 'cd ' + proj_path + '; git rev-parse HEAD'
        p = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)


        for line in p.stdout.readlines():
            print(line)
        retval = p.wait()
        proj_dict[row[0]]=(row[1],line)

print(proj_dict)


In [ ]:
pd.DataFrame(proj_dict.items())

In [ ]:
csv_file_path1 = os.path.join(BENCHMARK_PATH, "data/repo1.csv")
with open(csv_file_path1, "w") as f:
    w = csv.writer(f)
    w.writerows(proj_dict.items())

In [ ]:
csv_file_path1 = os.path.join(BENCHMARK_PATH, "data/repo1.csv")
with open(csv_file_path1, "w") as f:
    w = csv.writer(f)
    w.writerows(proj_dict.items())
